# 🧱 Structured extraction

Structured extraction asks a language model to fill in a record that we define, with one record
for each document. Delach et al. recorded the same few facts for each of 459 species: which species
it was, whether climate change was named as a threat, and whether any action to address it was
planned. A **schema** lists the fields of such a record and the values each field may take.

The [Laya example](laya-example.ipynb) codes one passage at a time and returns probabilities. Here
a language model reads a large part of a rule and returns a single record for the whole rule. The
model writes its answer as text, so every field is a claim that we check against the rule.

## Workflow

Extracting records from a corpus takes six steps, and each one is a choice we make and should
report.

1. **Corpus.** We download the full text of each listing rule from the Federal Register, as in the
   Laya example.
2. **Unit.** We extract one record per rule, because the questions we ask (which species, what
   the agency concluded) are about the rule as a whole.
3. **Context.** A rule runs to tens of thousands of words, more than we want to send for every
   record. We send the opening paragraphs, which include the summary, and the paragraphs that
   mention a climate-related term. Whatever we leave out, the model cannot use.
4. **Schema.** We write the fields, their types, and a description of each one.
5. **Extraction.** The model fills in the schema once for each rule.
6. **Audit.** We check the records against the text, with code where we can and by reading where
   we cannot.

## Schema

We write the schema as a Python class with [Pydantic](https://docs.pydantic.dev/). Each field has
a name, a type, and often a description. The type limits what the model may return.

- `list[str]` allows any number of free-text values, such as species names.
- `Literal[...]` allows exactly one value from a list that we write, like a choice question in
  Laya.
- `list[Literal[...]]` allows any number of values from that list.
- `bool` allows only true or false, like a noul question in Laya, but with no probability.
- `str` allows any text. We use it for a quotation that supports the answer, so that we can check
  the answer against the rule.

Field names, allowed values and descriptions are the only definitions the model has, so writing
them is the main work of designing a schema. Our schema records the two codes of Delach et al.,
a climate threat and an action to address it, together with the species and a quotation:

In [1]:
from typing import Literal
from pydantic import BaseModel, Field

class ListingRecord(BaseModel):
    species: list[str] = Field(
        description="scientific names of every species this rule lists")
    climate_role: Literal["threat to the species",
                          "considered, not a significant threat",
                          "not discussed"]
    mechanisms: list[Literal["sea level rise", "drought", "higher temperatures",
                             "altered precipitation", "storms", "wildfire", "other"]]
    climate_action: bool = Field(
        description="whether the rule describes an action to reduce a climate-related threat")
    evidence: str = Field(
        description="one sentence copied verbatim from the text that supports climate_role")

## The request

Each request has two parts: the **text** we send and the **schema**. LangChain converts the
Pydantic class into a [JSON Schema](https://json-schema.org/understanding-json-schema/about)
and sends it to the model as the definition of a tool. The model answers by calling that tool
with arguments that fill in the record, and LangChain checks the arguments against the schema
and returns a `ListingRecord`. Part of the record for our first rule, written as JSON, looks
like this:

```json
{"species": ["Plestiodon egregius egregius"],
 "climate_role": "threat to the species",
 "climate_action": false}
```

We use `deepseek-v4-flash` on the National Research Platform, the same model that Kilo Code
uses. The connection needs your `NRP_TOKEN`, which the [.env file page](env-file.md) shows how to
store. Two settings matter for this model. `method="function_calling"` asks for the record as a
tool call, which this model handles more reliably than the other methods, and turning off
*thinking* makes each answer faster. We also set the temperature to 0, so that the model chooses
its most likely answer each time.

## A rule

Our example is the same [final rule](https://www.federalregister.gov/d/2026-14636) used in the
Laya example, which listed the Florida Keys mole skink as endangered in July 2026. We send the
first two paragraphs of the rule, which hold the header and the summary, and the 17 paragraphs
that mention a climate-related term. The model's record for the rule:

In [2]:
import os
import re
import pathlib
import requests
import pandas as pd
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()
llm = ChatOpenAI(model="deepseek-v4-flash", temperature=0, timeout=180,
                 extra_body={"thinking": {"type": "disabled"}},
                 base_url="https://ellm.nrp-nautilus.io/v1", api_key=os.environ["NRP_TOKEN"])
extract = llm.with_structured_output(ListingRecord, method="function_calling")

pd.set_option("display.max_colwidth", None)

rules = pathlib.Path("rules")
rules.mkdir(exist_ok=True)

def download(document_number):
    path = rules / f"{document_number}.txt"
    if not path.exists():
        meta = requests.get(
            f"https://www.federalregister.gov/api/v1/documents/{document_number}.json",
            params={"fields[]": "raw_text_url"}, timeout=60).json()
        path.write_text(requests.get(meta["raw_text_url"], timeout=60).text)
    return path

def paragraphs(path):
    text = re.sub(r"<[^>]+>", "", path.read_text())
    paras = [re.sub(r"\s+", " ", p).strip() for p in re.split(r"\n\s*\n", text)]
    return [p for p in paras if len(p) > 200]

def context(path):
    paras = paragraphs(path)
    climate = [p for p in paras if re.search(r"climat|sea level|flood|storm|drought", p, re.I)]
    return "\n\n".join(dict.fromkeys(paras[:2] + climate))

skink = download("2026-14636")
record = extract.invoke(context(skink))
pd.Series(record.model_dump())

species                                                                                                                                                                                                                 [Plestiodon egregius egregius]
climate_role                                                                                                                                                                                                                     threat to the species
mechanisms                                                                                                                                                                                                             [sea level rise, storms, other]
climate_action                                                                                                                                                                                                                                   False
evidence    

The evidence sentence opens the paragraph we coded in the Laya example. The record agrees
with that paragraph that sea level rise and storms threaten the species. Laya's choice question
named development as the most important threat in the paragraph, but our schema does not ask
which threat is most important, so the two answers do not conflict.

## Audit

A record is a set of claims, and some of them can be checked with code. The `evidence` field
should be a sentence copied from the rule, so we can search for it in the text we sent. We search
twice: once in the text exactly as sent, and once after removing the page markers that the
Federal Register inserts (such as `[[Page 45690]]`) and treating every run of spaces and line
breaks as a single space.

In [3]:
def normalize(text):
    text = re.sub(r"<[^>]+>|\[\[Page \d+\]\]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def audit(path, record):
    return {"rule": path.stem,
            "in_context": record.evidence in context(path),
            "in_rule": normalize(record.evidence) in normalize(path.read_text())}

pd.Series(audit(skink, record))

rule          2026-14636
in_context          True
in_rule             True
dtype: object

Other fields need a reader. We can compare the species list with the rule's title and summary,
and we can read the paragraphs the model saw to decide whether we agree with `climate_role` and
`climate_action`.

## A folder of rules

A schema is written once and applied to every rule in a corpus. We add the rule that lists the
Peñasco least chipmunk, as in the Laya example, extract a record for each rule in the folder, and
audit each `evidence` quotation.

In [4]:
download("2024-28338")

rows = []
for path in sorted(rules.glob("*.txt")):
    record = extract.invoke(context(path))
    rows.append(record.model_dump() | audit(path, record))
records = pd.DataFrame(rows).set_index("rule")
records[["species", "climate_role", "mechanisms", "climate_action", "evidence",
         "in_context", "in_rule"]]

,species,climate_role,mechanisms,climate_action,evidence,in_context,in_rule
rule,,,,,,,
2024-28338,[Neotamias minimus atristriatus],"considered, not a significant threat","[drought, higher temperatures, altered precipitation, wildfire]",False,"Therefore, although the impacts of climate change on the Pe[ntilde]asco least chipmunk are expected to be negative, our analysis did not find it to be one of the most significant risk factors for the subspecies and its habitat over the next 30-year period.",False,True
2026-14636,[Plestiodon egregius egregius],threat to the species,"[sea level rise, storms, other]",False,"We have determined that the Florida Keys mole skink is endangered due to threats associated with sea level rise, increased high tide flooding, and increased-intensity storm events (Factor E), as well as threats due to habitat loss and degradation that result from development and habitat disturbance (Factor A).",True,True


The model calls climate change a threat for the skink and "considered, not a significant
threat" for the chipmunk. Laya's average codes in the Laya example pointed the same way (0.44 and
0.17 on the yes or no threat question), but here the difference is a single stated conclusion,
and the chipmunk's evidence sentence is that conclusion. Neither record finds an action to reduce
a climate-related threat.

Both quotations are in the rule, but the chipmunk's fails the first search. A page break falls in
the middle of that sentence, so in the text we sent it is split across two paragraphs. A check
that is too strict reports a correct quotation as invented, and one that is too loose accepts a
paraphrase, so the check itself needs testing.

The skink record also quotes a different sentence from the one in the first record, although the
model saw the same text with the temperature set to 0. Both sentences support the same answer.
A temperature of 0 does not guarantee identical answers from a model served to many users, so we
save each record when we first extract it instead of extracting it again.

## Questions

- Which fields would you trust across all of the listing rules, and how would you check the ones
  you would not?
- What can the model not see in the context we send? Find a rule where a paragraph we left out
  would change a field.
- Laya gives a probability for each passage, and extraction gives one answer for each rule. For
  each step of your analysis, which would you use, and why?
- Change one allowed value of `climate_role`, for example by splitting "threat to the species"
  into a main threat and a minor threat. How do the records change?

The code for each step is folded above each result. The LangChain guide to
[structured outputs](https://python.langchain.com/docs/concepts/structured_outputs/) and the
[Pydantic documentation](https://docs.pydantic.dev/) describe the full interface.